In [63]:
# ==================== FUNCTION DEFINITION =====================


### Function to split the box, if it is too big (max. 2500px): ###
def split_bbox(bbox, max_width, max_height, resolution):
    from shapely.geometry import box
    from sentinelhub import BBox, CRS

    bbox_geom = box(*bbox)
    minx, miny, maxx, maxy = bbox

    width_m = maxx - minx
    height_m = maxy - miny

    tile_width = resolution * max_width
    tile_height = resolution * max_height

    tiles = []
    y = miny
    while y < maxy:
        x = minx
        while x < maxx:
            tile_maxx = min(x + tile_width, maxx)
            tile_maxy = min(y + tile_height, maxy)
            tile = BBox([x, y, tile_maxx, tile_maxy], crs=CRS(32632))
            tiles.append(tile)
            x += tile_width
        y += tile_height
    return tiles

### Function to download the FSC data from copernicus browser API ###
# define variables
def process_fsc_data(coords: [float, float, float, float],
                     selected_dates: list,
                     output_folder: str,
                     time_interval: [str, str],
                     eval_fsc_scl_mask,
                     glacier_shp: str,
                     lake_shp: str,
                     forest_mask_path: str,
                     resolution):
    # import utilities
    import os
    import shutil
    from pathlib import Path
    from datetime import datetime, timedelta

    import numpy as np
    import pandas as pd
    import geopandas as gpd
    import rioxarray as rxr
    import xarray as xr
    import re

    from rasterio.features import rasterize
    from sentinelhub import (
        SHConfig, DataCollection, SentinelHubCatalog, SentinelHubRequest,
        MimeType, BBox, bbox_to_dimensions, CRS
    )
    
    # set up the connection!
    config = SHConfig("MO_v2")
    if not config.sh_client_id or not config.sh_client_secret:
        raise RuntimeError("SentinelHub credentials missing!")
    else: 
        print("Successfully connected to Copenicus API", "\n")
    
    # set crs, bbox
    epsg = 32632
    aoi_bbox = BBox(bbox=coords, crs=CRS(epsg))
    aoi_size = bbox_to_dimensions(aoi_bbox, resolution=resolution)
    print(f"Image shape at {resolution} m resolution: {aoi_size} pixels")

    # Check if size exceeds 2500x2500
    if aoi_size[0] > 2500 or aoi_size[1] > 2500:
        print("AOI exceeds max size, splitting into tiles...")
        tile_bboxes = split_bbox(coords, max_width=2500, max_height=2500, resolution=resolution)
    else:
        tile_bboxes = [aoi_bbox]

    # Print the sizes of the tiles
    for i, tile in enumerate(tile_bboxes):
        tile_size = bbox_to_dimensions(tile, resolution=resolution)
        print(f"Tile {i}: {tile_size[0]} x {tile_size[1]} pixels")
        
    print("\n")
    
    catalog = SentinelHubCatalog(config=config)
    search_iterator = catalog.search(
        DataCollection.SENTINEL2_L2A,
        bbox=aoi_bbox,
        time=time_interval,
        fields={"include": ["id", "properties.datetime"], "exclude": []},
        filter="eo:cloud_cover < 50"
    )

    results = list(search_iterator)
    print("Number of available dates:", len(results))

    datetime_list = sorted({datetime.strptime(result["properties"]["datetime"][:10], '%Y-%m-%d') for result in results})
    # print available dates
    print("Available dates:", ', '.join(d.strftime("%Y-%m-%d") for d in datetime_list))

    # Download images
    ndsi_images = []
    for current_date in datetime_list:
        time_interval_day = (current_date.strftime("%Y-%m-%d"), current_date.strftime("%Y-%m-%d"))
        print(f"Downloading image for: {time_interval_day[0]}")

        for i, tile in enumerate(tile_bboxes):
            print(f"           Tile {i}")
            tile_size = bbox_to_dimensions(tile, resolution=resolution)
            tile_folder = os.path.join(output_folder, f"{time_interval_day[0]}_tile_{i}")

            request_ndsi = SentinelHubRequest(
                data_folder=tile_folder,
                evalscript=eval_fsc_scl_mask,
                input_data=[
                    SentinelHubRequest.input_data(
                        data_collection=DataCollection.SENTINEL2_L2A.define_from("s2l2a", service_url=config.sh_base_url),
                        time_interval=time_interval_day
                    )
                ],
                responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
                bbox=tile,
                size=tile_size,
                config=config,
            )
            ndsi_imgs = request_ndsi.get_data(save_data=True)
            ndsi_images.extend(ndsi_imgs)


    # Reorganize folder structure
    def process_tiff_files(base_path, resolution):
        from collections import defaultdict
        import rioxarray as rxr
        import xarray as xr
        import shutil
        from pathlib import Path

        base_path = Path(base_path)
        date_tile_groups = defaultdict(list)

        # Step 1: Reorganize and collect tile files
        for date_folder in base_path.iterdir():
            if date_folder.is_dir():
                nested = list(date_folder.glob("*/response.tiff"))
                if len(nested) == 1:
                    tile_id = date_folder.name.split('_tile_')[-1]
                    date_part = date_folder.name.split('_tile_')[0]
                    new_name = f"ndsi_{date_part}_tile_{tile_id}.tiff"
                    target_path = base_path / new_name
                    shutil.move(str(nested[0]), target_path)
                    shutil.rmtree(date_folder)
                    print(f"Moved and renamed {new_name}")
                    date_tile_groups[date_part].append(target_path)

        print("Folder structure reorganized. Next merging tiles to one single file...")

        # Step 2: Merge tiles per date
        for date, file_list in date_tile_groups.items():
            import rasterio
            from rasterio.merge import merge as rio_merge

            # Read raster files as datasets
            src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]

            # Merge them using rasterio
            mosaic, out_transform = rio_merge(src_files_to_mosaic)

            # Copy metadata and update it for merged output
            out_meta = src_files_to_mosaic[0].meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": mosaic.shape[1],
                "width": mosaic.shape[2],
                "transform": out_transform
            })

            # Write to disk
            out_path = base_path / f"ndsi_{date}.tiff"
            with rasterio.open(out_path, "w", **out_meta) as dest:
                dest.write(mosaic)

            # Close all source files
            for src in src_files_to_mosaic:
                src.close()

            print(f"Merged tiles and saved: {out_path.name}")

            # Optionally remove individual tile files
            for fp in file_list:
                fp.unlink()

        print("All tiles merged per date to one file.")
    
    process_tiff_files(output_folder, resolution=resolution)

    # Combine datasets
    pattern = re.compile(r"ndsi_(\d{4}-\d{2}-\d{2})\.tiff")
    datasets = []
    for file in os.listdir(output_folder):
        if pattern.match(file):
            date_str = pattern.match(file).group(1)
            date = pd.to_datetime(date_str).date()
            dataset = rxr.open_rasterio(os.path.join(output_folder, file)).expand_dims(time=[pd.Timestamp(date)])
            datasets.append(dataset)

    if not datasets:
        print("No datasets found!")
        return

    combined_dataset = xr.concat(datasets, dim="time")
    combined_dataset.rio.write_nodata(np.nan, inplace=True)

    # Filter based on NaN coverage
    nan_fraction = combined_dataset.isnull().sum(dim=["x", "y"]) / (combined_dataset.sizes["x"] * combined_dataset.sizes["y"])
    valid_indices = (nan_fraction <= 0.70).values.flatten()
    combined_dataset = combined_dataset.isel(time=valid_indices)

    print(f"Datasets filtered to exclude very high cloud cover")

    # Prepare folders
    selected_dir = os.path.join(output_folder, "selected")
    masked_dir = os.path.join(output_folder, "selected_masked")
    os.makedirs(masked_dir, exist_ok=True)

    if selected_dates:
        os.makedirs(selected_dir, exist_ok=True)
        selected_date_strs = set(dt.strftime("%Y-%m-%d") for dt in selected_dates)

        for file in os.listdir(output_folder):
            if file.startswith("ndsi_") and file.endswith(".tiff"):
                if file[5:-5] in selected_date_strs:
                    shutil.copy2(os.path.join(output_folder, file), os.path.join(selected_dir, file))
                    print(f"Copied {file} to selected folder")

        files_to_mask = [f for f in os.listdir(selected_dir) if f.endswith(".tiff") and f.startswith("ndsi_")]
    else:
        print("No selected dates provided. Skipping selection. Masking all available NDSI images.\n")
        files_to_mask = [f for f in os.listdir(output_folder) if f.endswith(".tiff") and f.startswith("ndsi_")]

    # Load mask data
    glacier_gdf = gpd.read_file(glacier_shp)
    lake_gdf = gpd.read_file(lake_shp)
    forest_mask = rxr.open_rasterio(forest_mask_path).squeeze()

    # Apply masks
    for file in files_to_mask:
        img_path = os.path.join(selected_dir if selected_dates else output_folder, file)
        img = rxr.open_rasterio(img_path).squeeze()
        forest_mask_matched = forest_mask.rio.reproject_match(img)

        glacier_mask = rasterize(
            [(geom, 1) for geom in glacier_gdf.to_crs(img.rio.crs).geometry],
            out_shape=img.shape,
            transform=img.rio.transform(),
            fill=0,
            dtype="uint8"
        )

        lake_mask = rasterize(
            [(geom, 1) for geom in lake_gdf.to_crs(img.rio.crs).geometry],
            out_shape=img.shape,
            transform=img.rio.transform(),
            fill=0,
            dtype="uint8"
        )

        combined_mask = (forest_mask_matched.values == 1) | (glacier_mask == 1) | (lake_mask == 1)
        masked_img = img.where(~combined_mask)

        masked_img.rio.to_raster(os.path.join(masked_dir, file))
        print(f"Masked and saved: {file}")


In [64]:
# define coordinates for the bbox
coords_achensee=(688675, 5250625, 710875, 5274625)
coords_kuehtai=(645975, 5204025, 673175, 5234725)
coords_kaunertal=(618775, 5187025, 646775, 5213325)

coords_engadin=(550975, 5131625,  613475, 5198125)
coords_ziller=(676275, 5205325, 738375, 5260925)
coords_nordost=(688675, 5239325, 749975, 5286925)
coords_zentral=(583375, 5180625,  701875, 5249625)

# Define all selected dates per AOI and season
from datetime import datetime

selected_dates_kuehtai_22_23 = [
    datetime(2022, 10, 3), datetime(2022, 10, 28), datetime(2023, 3, 22),
    datetime(2023, 6, 7), datetime(2023, 6, 20), datetime(2023, 6, 25),
    datetime(2023, 7, 15), datetime(2023, 8, 11)
]

selected_dates_kuehtai_23_24 = [
    datetime(2023, 10, 13), datetime(2023, 10, 28), datetime(2024, 2, 5),
    datetime(2024, 3, 11), datetime(2024, 4, 12), datetime(2024, 5, 20),
    datetime(2024, 6, 19), datetime(2024, 6, 29), datetime(2024, 7, 9),
    datetime(2024, 7, 14), datetime(2024, 8, 10)
]

selected_dates_kaunertal_22_23 = [
    datetime(2022, 10, 3), datetime(2022, 10, 28), datetime(2023, 3, 22),
    datetime(2023, 5, 3), datetime(2023, 6, 7), datetime(2023, 6, 25),
    datetime(2023, 7, 15), datetime(2023, 8, 11)
]

selected_dates_kaunertal_23_24 = [
    datetime(2023, 10, 25), datetime(2023, 10, 28), datetime(2024, 2, 5),
    datetime(2024, 3, 11), datetime(2024, 4, 12), datetime(2024, 5, 20),
    datetime(2024, 6, 19), datetime(2024, 6, 29), datetime(2024, 7, 9),
    datetime(2024, 7, 14), datetime(2024, 8, 10)
]

selected_dates_achensee_22_23 = [
    datetime(2022, 11, 9), datetime(2022, 11, 27), datetime(2023, 1, 1),
    datetime(2023, 2, 10), datetime(2023, 2, 15), datetime(2023, 2, 22),
    datetime(2023, 3, 7), datetime(2023, 3, 22), datetime(2023, 4, 21)
]

selected_dates_achensee_23_24 = [
    datetime(2023, 11, 4), datetime(2023, 12, 27), datetime(2024, 1, 3),
    datetime(2024, 2, 15), datetime(2024, 3, 8), datetime(2024, 4, 7),
    datetime(2024, 4, 27), datetime(2024, 4, 30)
]

# define the eval script
eval_fsc_scl_mask = """
//VERSION=3
function setup() {
  return {
    input: ["B03", "B11", "SCL"],
    output: { bands: 1, sampleType: "FLOAT32" }
  };
}

function evaluatePixel(sample) {
  let ndsi = (sample.B03 - sample.B11) / (sample.B03 + sample.B11);
  let image_mask = sample.SCL;

  // Mask values: adjust as needed
  let mask_values = [1,6,8,9];

  // FSC thresholds
  let ndsi_min = 0.1;
  let ndsi_max = 0.6;

  // Calculate FSC
  let fsc = 0.0;
  if (ndsi <= ndsi_min) {
    fsc = 0.0;
  } else if (ndsi >= ndsi_max) {
    fsc = 1.0;
  } else {
    fsc = (ndsi - ndsi_min) / (ndsi_max - ndsi_min);
  }

  // Mask out unwanted pixels
  let fsc_masked = mask_values.includes(image_mask) ? NaN : fsc;

  return [fsc_masked];
}
"""

In [68]:
# Example use for ziller 2018-2019
process_fsc_data(
    coords= coords_ziller,
    resolution=20,
    selected_dates= [datetime(2020, 10, 18)],
    output_folder=r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\FSC_data\zillerrr",
    time_interval=("2020-10-01", "2020-10-20"),
    eval_fsc_scl_mask=eval_fsc_scl_mask,
    glacier_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\GI_5\Gletscher_AOIs.shp",
    lake_shp = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\shapefiles\Seen_AOIs.shp",
    forest_mask_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data\ses_topo_22_forest_mask_domain_epsg32632.tif"
    )

Successfully connected to Copenicus API 

Image shape at 20 m resolution: (3105, 2780) pixels
AOI exceeds max size, splitting into tiles...
Tile 0: 2500 x 2500 pixels
Tile 1: 605 x 2500 pixels
Tile 2: 2500 x 280 pixels
Tile 3: 605 x 280 pixels


Number of available dates: 2
Available dates: 2020-10-08, 2020-10-18
           Tile 0
           Tile 1
           Tile 2
           Tile 3
           Tile 0
           Tile 1
           Tile 2
           Tile 3
Moved and renamed ndsi_2020-10-08_tile_0.tiff
Moved and renamed ndsi_2020-10-08_tile_1.tiff
Moved and renamed ndsi_2020-10-08_tile_2.tiff
Moved and renamed ndsi_2020-10-08_tile_3.tiff
Moved and renamed ndsi_2020-10-18_tile_0.tiff
Moved and renamed ndsi_2020-10-18_tile_1.tiff
Moved and renamed ndsi_2020-10-18_tile_2.tiff
Moved and renamed ndsi_2020-10-18_tile_3.tiff
Folder structure reorganized. Next merging tiles to one single file...
Merged tiles and saved: ndsi_2020-10-08.tiff
Merged tiles and saved: ndsi_2020-10-18.tiff
All tiles me